## Imoorts

In [14]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import json
import cv2
import mediapipe as mp
import pandas as pd
import os
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from animation_tools import *
from IPython.display import HTML
import joblib

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

## Functions for pipeline

In [33]:

def extract_mediapipe_to_csv(data_path: str, save_path: str, model_path: str):

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]
    static = not any(data_path.lower().endswith(ext) for ext in video_extensions)

    base_options = python.BaseOptions(model_asset_path=model_path)

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,  
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    data = []

    if static:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            image = cv2.imread(data_path)
            if image is None:
                raise ValueError(f"Could not read image: {data_path}")

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            results = landmarker.detect(mp_image)

            if results.pose_landmarks:
                row = {"FrameNo": 0}

                for joint in JOINT_ORDER:
                    idx = LANDMARK_INDEX[joint]
                    lm = results.pose_landmarks[0][idx]

                    row[f"{joint}_x"] = lm.x
                    row[f"{joint}_y"] = lm.y
                    row[f"{joint}_z"] = lm.z

                data.append(row)

    else:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            cap = cv2.VideoCapture(data_path)

            if not cap.isOpened():
                raise ValueError(f"Could not open video: {data_path}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                timestamp_ms = int((frame_idx / fps) * 1000)
                results = landmarker.detect_for_video(mp_image, timestamp_ms)

                if results.pose_landmarks:
                    row = {"FrameNo": frame_idx}

                    for joint in JOINT_ORDER:
                        idx = LANDMARK_INDEX[joint]
                        lm = results.pose_landmarks[0][idx]

                        row[f"{joint}_x"] = lm.x
                        row[f"{joint}_y"] = lm.y
                        row[f"{joint}_z"] = lm.z

                    data.append(row)

                frame_idx += 1

            cap.release()

    # ---- SAVE CSV ----
    columns = ["FrameNo"]
    for joint in JOINT_ORDER:
        columns += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    df = pd.DataFrame(data)
    df = df[columns]

    return df

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)

    print(f"Saved {len(df)} rows to {save_path}")



def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None
    

class MovementClassifier(nn.Module):
    """
    Sequence-to-sequence binary classifier.
    Input:  (batch, seq_len, 39)  -- 13 joints x 3 coords
    Output: (batch, seq_len, 1)   -- logit per frame (use BCEWithLogitsLoss)
    """
    def __init__(self, hidden_layers: list, layer_type="LSTM", dropout=0.0):
        super().__init__()
        self.layer_type = layer_type

        input_size = 39  # 13 joints x 3 (x, y, z)
        rnn_class = nn.LSTM if layer_type == "LSTM" else nn.GRU

        self.rnns  = nn.ModuleList()
        self.drops = nn.ModuleList()

        sizes = [input_size] + hidden_layers
        for i in range(len(hidden_layers)):
            self.rnns.append(rnn_class(sizes[i], sizes[i + 1], batch_first=True))
            self.drops.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())

        self.fc_out = nn.Linear(hidden_layers[-1], 1)  # 1 logit per frame

    def forward(self, x):
        for rnn, drop in zip(self.rnns, self.drops):
            x, _ = rnn(x)
            x = drop(x)
        return self.fc_out(x)  # (batch, seq_len, 1)


def trim_csv_with_rnn_model(model, df, scaler, seq_length=30, stride=15, threshold=0.5):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "FrameNo" not in df.columns:
        df["FrameNo"] = np.arange(len(df))

    feature_cols = [c for c in df.columns
        if c.endswith("_x") or c.endswith("_y") or c.endswith("_z")]

    X_np = df[feature_cols].values.astype(np.float32)
    X_scaled = scaler.transform(X_np)

    probs_sum = np.zeros(len(df))
    counts = np.zeros(len(df))

    model.eval()
    with torch.no_grad():
        for i in range(0, len(df) - seq_length + 1, stride):
            seq = X_scaled[i:i + seq_length]

            X = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            logits = model(X)

            probs = torch.sigmoid(logits).cpu().numpy().flatten()

            probs_sum[i:i + seq_length] += probs
            counts[i:i + seq_length] += 1

    frame_probs = probs_sum / np.maximum(counts, 1)
    preds = (frame_probs >= threshold).astype(int)

    print("Predictions:")
    print(preds)

    df["pred_running"] = preds

    movement_frames = df[df["pred_running"] == 1]

    if len(movement_frames) == 0:
        print("No movement detected")
        return None

    start_idx = movement_frames.index.min()
    stop_idx = movement_frames.index.max()

    trimmed_df = df.loc[start_idx:stop_idx].drop(columns=["pred_running"])

    print(f"Start index: {start_idx}, Stop index: {stop_idx}")

    return trimmed_df

## Import video

In [16]:
video_path = "../data/full_squat_cut.mov"


## Run mediapipe on video

In [17]:
df_mediapipe = extract_mediapipe_to_csv(
                                    data_path=video_path,
                                    save_path="",
                                    model_path="../data/pose_landmarker.task"
                                    )

df_mediapipe = df_mediapipe.drop(columns=["FrameNo"])

print(df_mediapipe)

I0000 00:00:1777885483.380196 3533351 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1777885483.577884 3559091 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777885483.619471 3559088 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


       head_x    head_y    head_z  left_shoulder_x  left_shoulder_y  \
0    0.344205  0.578638 -0.078560         0.406801         0.457659   
1    0.343457  0.576607 -0.083377         0.405961         0.457602   
2    0.343441  0.571190 -0.083162         0.405957         0.452893   
3    0.343126  0.563026 -0.076876         0.406170         0.446547   
4    0.343137  0.557325 -0.076762         0.406346         0.439840   
..        ...       ...       ...              ...              ...   
171  0.338445  0.540009 -0.075714         0.406842         0.431980   
172  0.338389  0.539989 -0.089836         0.407179         0.431179   
173  0.338302  0.539993 -0.116898         0.407682         0.430033   
174  0.338303  0.539539 -0.116523         0.407872         0.428396   
175  0.336955  0.539460 -0.110143         0.407781         0.428323   

     left_shoulder_z  left_elbow_x  left_elbow_y  left_elbow_z  \
0           0.027580      0.496694      0.426329      0.046188   
1           0.0

## Call  trained model for to detect movement frames

In [18]:
metadata_path = "../../MainProject/Assignment11/binary_classificator_models/recurrant_classification"
model_path = "../../MainProject/Assignment11/binary_classificator_models/recurrant_classification/champion_model.pt"


champion_info = load_champion_info(metadata_dir=metadata_path)

config = champion_info["hyperparameters"]

model = MovementClassifier(
    hidden_layers=config["hidden_layers"],
    layer_type=config.get("layer_type", "LSTM"),
    dropout=config["dropout"]
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))

model.eval()

MovementClassifier(
  (rnns): ModuleList(
    (0): LSTM(39, 160, batch_first=True)
    (1): LSTM(160, 160, batch_first=True)
    (2): LSTM(160, 64, batch_first=True)
  )
  (drops): ModuleList(
    (0-2): 3 x Dropout(p=0.4, inplace=False)
  )
  (fc_out): Linear(in_features=64, out_features=1, bias=True)
)

## Use trained model to detect movement frames and trim frames

In [35]:
scaler_path = "../../MainProject/Assignment11/binary_classificator_models/recurrant_classification/scaler.pkl"
scaler = joblib.load(scaler_path)



df_mediapipe_trimmed = trim_csv_with_rnn_model(
                                            model=model,
                                            df=df_mediapipe,
                                            scaler=scaler,
                                            seq_length=30,   # match training
                                            stride=15,       # or 30 if that’s what you trained with
                                            threshold=0.5
)

print(df_mediapipe_trimmed)


Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
No movement detected
None


## create skeleton animation

In [ ]:
anim = animate_df(
    df_mediapipe_trimmed,
    save_folder_path="../plots",
    output_file="mediapipe_trimmed.gif"
)

HTML(anim.to_jshtml())

AttributeError: 'NoneType' object has no attribute 'copy'